In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import numpy as np
import nltk
from nltk.tokenize import word_tokenize, RegexpTokenizer, TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer as wnl
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize lemmatizer once
lemmatizer = wnl()

In [ ]:
df = pd.read_csv('../data/dcInbox/dcinbox_export_114.csv')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()
df = df[pd.to_numeric(df['Unix Timestamp'], errors='coerce').notna()].copy()
df['datetime'] = pd.to_datetime(df['Unix Timestamp'], unit='ms')

# Sort chronologically
df = df.sort_values('datetime').reset_index(drop=True)

# Filter data for February and March
# df = df[df['datetime'].between('2016-01-01', '2016-12-31')]
df = df[df['Chamber'] == 'House']
df.info()

republican_emails = df[df['Party'] == 'Republican']
democrat_emails = df[df['Party'] == 'Democrat']

In [ ]:
# ENHANCED CUSTOM STOPWORDS FOR POLITICAL TEXT
# Focus on removing truly non-political administrative/technical terms

# Load NLTK stopwords
try:
    nltk_stopwords = set(stopwords.words('english'))
except LookupError:
    nltk_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 
                      'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 
                      'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 
                      'would', 'could', 'should'}

# Custom stopwords - ONLY truly generic/administrative terms
custom_stopwords = {
    # Email/web formatting
    'ha', 'wa', 'click', 'email', 'link', 'phone', 'fax', 'pm', 'am', 'window',
    'plain', 'version', 'text', 'website', 'browser', 'facebook', 'twitter', 
    'instagram', 'youtube', 'unsubscribe', 'subscribe', 'newsletter', 'san',
    
    # Email salutations/closings (non-political)
    'dear', 'sincerely', 'regard', 'regards', 'thank', 'thanks', 'best',
    'cordially', 'respectfully',
    
    # Generic congressional administrative (non-policy)
    'representative', 'rep', 'office', 'district', 'constituent', 'contact',
    'washington', 'capitol', 'room', 'building', 'floor', 'street', 'avenue',
    
    # Generic verbs/actions (truly meaningless)
    'thing', 'things', 'way', 'ways', 'use', 'used', 'make', 'look', 'want',
    'come', 'coming', 'going', 'took', 'spoke', 'heard', 'said', 'told',
    'need', 'needs', 'like', 'good', 'great', 'sure', 'follow', 'stop',
    'mean', 'means', 'real', 'clear', 'big', 'better', 'best',
    'give', 'get', 'got', 'see', 'let', 'take', 'took', 'put', 'bring',
    'keep', 'kept', 'send', 'sent', 'receive', 'received', 'grant',
    
    # Time references (generic)
    'ago', 'earlier', 'later', 'nearly', 'instead', 'currently', 'recent',
    'today', 'yesterday', 'tomorrow', 'week', 'month', 'year', 'day', 'weekend',
    'weekly', 'monthly', 'yearly', 'daily', 'hourly', 'minutes', 'seconds',
    'rsvp', 'attend', 'attending', 'attends', 'attending', 'attends', 'attending',
    
    # Administrative procedural (non-substantive)
    'update', 'updates', 'information', 'inform', 'share', 'sharing',
    'provide', 'providing', 'ensure', 'allow', 'help', 'helping', 'working',
    'please', 'welcome', 'join', 'invite', 'forward', 'forwarding',
    
    # State abbreviations (if not meaningful for analysis)
    'st', 'rd', 'ave', 'dr', 'pa', 'ny', 'ca', 'tx', 'fl', 'va', 'nc',
    'dc', 'al', 'ak', 'az', 'ar', 'co', 'ct', 'de', 'ga', 'hi', 'id',
    'il', 'ia', 'ks', 'ky', 'la', 'me', 'md', 'ma', 'mi', 'mn', 'ms',
    'mo', 'mt', 'ne', 'nv', 'nh', 'nj', 'nm', 'nd', 'oh', 'ok', 'or',
    'ri', 'sc', 'sd', 'tn', 'ut', 'vt', 'wa', 'wv', 'wi', 'wy',
    
    # Contractions/fragments
    'im', 'ive', 'dont', 'thats', 'youre', 'theyre', 'weve', 'theyve',
    'cant', 'wont', 'didnt', 'doesnt', 'wasnt', 'werent', 'hasnt', 'havent',
    
    # Numbers/dates
    '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', 
    '21', '22', '23', '24', '25', '30', '50', '100', '200', '300', '500',
    '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023',
    '20515',  # Common zip code in congressional emails
    
    # Meaningless fragments
    '___', 'cid', 'doe', 'ni', 'isi', 'hob', 'na', 'nbsp',
    
    # Generic personal names (common first names that appear frequently)
    'john', 'tom', 'bob', 'mike', 'doug', 'ryan', 'scott', 'jim', 'joe',
    'bill', 'dan', 'dave', 'steve', 'mark', 'paul', 'david', 'robert',
    
    # Non-political communication verbs
    'read', 'know', 'say', 'feel', 'think', 'believe', 'hope', 'wish',

    'house', 'congress', 'congressman', 'congresswoman', 'senator',
    'senate', 'member', 'members', 'colleague', 'colleagues',
    'legislation', 'bill', 'act', 'committee', 'hearing',
    'vote', 'voted', 'voting', 'passed', 'floor', 'session',
    'suite', 'room', 'office', 'building', 'capitol',
    'would', 'could', 'should', 'must', 'may', 'also', 'many',
    'one', 'first', 'last', 'every', 'back', 'new',
    'open', 'view', 'visit', 'list', 'image', 'removed',
    'privacy', 'privacy policy', 'news', 'message', 'medium',
    'question', 'address', 'call', 'meeting', 'event', 'hour',
    'congressional', 'member congress', 'united state',
}

# Additional domain-specific stopwords for congressional emails
congressional_stopwords = {
    # Standard congressional communication (non-policy)
    'congress', 'congressman', 'congresswoman', 'senator', 'house', 'senate',
    'member', 'members', 'colleague', 'colleagues',
    
    # Generic procedural (unless you want to track procedure itself)
    'vote', 'voted', 'voting', 'introduce', 'introduced', 'bill', 'legislation',
    'committee', 'hearing', 'testimony', 'session', 'floor',
    
    # Only add these if they're TOO common and not meaningful for your analysis
    # Be careful - some of these might actually be important!
    # 'support', 'oppose', 'work', 'fight', 'stand', 'proud'
}

# Combine all stopwords
# Note: You can toggle congressional_stopwords on/off depending on your needs
all_stopwords = nltk_stopwords | custom_stopwords  # | congressional_stopwords

print(f"Total stopwords: {len(all_stopwords)}")

In [ ]:
def tokenize_for_vectorizer(text):
    """
    Enhanced tokenizer for TF-IDF vectorizer.
    - Lemmatization for normalization
    - Filters numbers and very short words
    - Preserves political terminology
    """
    if not isinstance(text, str):
        text = str(text) if text is not None else ''
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text.strip())
    
    # Tokenize
    tokenizer = TweetTokenizer()
    words = tokenizer.tokenize(text.lower())
    
    # Clean tokens
    words = [re.sub(r"[^\w]", '', word) for word in words]  # Remove punctuation
    words = [word.lower() for word in words]  # Lowercase
    words = [word for word in words if word]  # Remove empty strings
    # remove times like 9:00 am
    words = [word for word in words if not re.match(r'^\d{1,2}:\d{2} (am|pm)$', word)]

    # Filter out pure numbers
    words = [word for word in words if not word.isdigit()]
    
    # Filter out very short words (keep 3+ characters)
    # Exception: keep common 2-letter abbreviations that might be political
    political_exceptions = {'us', 'un', 'eu', 'uk', 'gop', 'crt', 'dei'}
    words = [word for word in words if len(word) >= 3 or word in political_exceptions]
    
    return words

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,
    stop_words=list(all_stopwords),
    lowercase=True,
    tokenizer=tokenize_for_vectorizer,
    min_df=2,              # CHANGED from 5 to 2 - catches rarer phrases
    max_df=0.75,           # CHANGED from 0.85 - more aggressive filtering
    ngram_range=(1, 4),    # CHANGED to include 4-grams like "lock her up"
    sublinear_tf=True,
)

print("TF-IDF Vectorizer configured")
print(f"  - Max features: {tfidf_vectorizer.max_features}")
print(f"  - Min doc frequency: {tfidf_vectorizer.min_df}")
print(f"  - Max doc frequency: {tfidf_vectorizer.max_df}")
print(f"  - N-gram range: {tfidf_vectorizer.ngram_range}")

In [ ]:
def extract_top_tfidf_terms(corpus, vectorizer, top_n=100):
    """
    Extract top terms by TF-IDF score from a corpus.
    
    Parameters:
    - corpus: list of text documents
    - vectorizer: fitted TfidfVectorizer
    - top_n: number of top terms to return
    
    Returns:
    - DataFrame with terms and their mean TF-IDF scores
    """
    # Fit and transform
    tfidf_matrix = vectorizer.fit_transform(corpus)
    
    # Get feature names
    feature_names = vectorizer.get_feature_names_out()
    
    # Calculate mean TF-IDF score for each term across all documents
    mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
    
    # Create DataFrame
    tfidf_df = pd.DataFrame({
        'term': feature_names,
        'mean_tfidf': mean_tfidf
    }).sort_values('mean_tfidf', ascending=False)
    
    return tfidf_df.head(top_n)


def compare_party_terms(rep_corpus, dem_corpus, vectorizer, top_n=100):
    """
    Compare TF-IDF scores between Republican and Democratic corpora.
    Identifies terms that are distinctively used by each party.
    
    Parameters:
    - rep_corpus: list of Republican text documents
    - dem_corpus: list of Democratic text documents  
    - vectorizer: TfidfVectorizer instance
    - top_n: number of top distinctive terms per party
    
    Returns:
    - DataFrame with terms and their relative scores
    """
    # Fit on combined corpus to get same vocabulary
    combined_corpus = rep_corpus + dem_corpus
    vectorizer.fit(combined_corpus)
    
    # Transform each party's corpus
    rep_tfidf = vectorizer.transform(rep_corpus)
    dem_tfidf = vectorizer.transform(dem_corpus)
    
    # Get feature names
    feature_names = vectorizer.get_feature_names_out()
    
    # Calculate mean TF-IDF for each party
    rep_mean = np.asarray(rep_tfidf.mean(axis=0)).flatten()
    dem_mean = np.asarray(dem_tfidf.mean(axis=0)).flatten()
    
    # Calculate difference (positive = more Republican, negative = more Democratic)
    diff = rep_mean - dem_mean
    
    # Create comparison DataFrame
    comparison_df = pd.DataFrame({
        'term': feature_names,
        'rep_tfidf': rep_mean,
        'dem_tfidf': dem_mean,
        'difference': diff,
        'ratio': np.where(dem_mean > 0, rep_mean / dem_mean, rep_mean * 100)
    })
    
    # Get top Republican terms
    rep_distinctive = comparison_df.nlargest(top_n, 'difference')
    
    # Get top Democratic terms  
    dem_distinctive = comparison_df.nsmallest(top_n, 'difference')
    
    return rep_distinctive, dem_distinctive, comparison_df

def visualize_top_terms(tfidf_df, top_n=30, title="Top Terms by TF-IDF Score"):
    """
    Create a bar plot of top terms.
    """
    plt.figure(figsize=(12, 8))
    
    top_terms = tfidf_df.head(top_n)
    
    plt.barh(range(len(top_terms)), top_terms['mean_tfidf'])
    plt.yticks(range(len(top_terms)), top_terms['term'])
    plt.xlabel('Mean TF-IDF Score')
    plt.title(title)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# Prepare corpora
rep_df = df[df['Party'] == 'Republican'].copy()
rep_df['full_text'] = (rep_df['Subject'].fillna('').astype(str) + ' ' + 
                       rep_df['Body'].fillna('').astype(str)).str.lower()
rep_corpus = rep_df['full_text'].tolist()

dem_df = df[df['Party'] == 'Democrat'].copy()
dem_df['full_text'] = (dem_df['Subject'].fillna('').astype(str) + ' ' + 
                       dem_df['Body'].fillna('').astype(str)).str.lower()
dem_corpus = dem_df['full_text'].tolist()

print("="*70)
print("FINDING DISTINCTIVE REPUBLICAN HOT BUTTON TERMS")
print("="*70)

# USE COMPARATIVE ANALYSIS - this is what you need!
rep_distinctive, dem_distinctive, full_comparison = compare_party_terms(
    rep_corpus, dem_corpus, tfidf_vectorizer, top_n=200
)

# Show terms that are DISTINCTIVELY Republican
print("\nTop 50 Distinctive Republican Terms:")
print("(High difference = more uniquely Republican)")
print("-"*70)

for i, row in rep_distinctive.head(200).iterrows():
    print(f"{row['term']:35s} | Rep: {row['rep_tfidf']:.4f} | Dem: {row['dem_tfidf']:.4f} | Diff: {row['difference']:.4f}")

In [ ]:
for i, row in dem_distinctive.head(200).iterrows():
    print(f"{row['term']:35s} | Dem: {row['dem_tfidf']:.4f} | Rep: {row['rep_tfidf']:.4f} | Diff: {row['difference']:.4f}")